In the previous set of notebooks we explored several ways of constructing a CAD
in SQL. With the column-based non-recursive approach being the fastest, we'll
use that one to now perform quantifier elimination on LRA formulas.

## Quantifier Elimination

Suppose we have an $n$-dimensional CAD for an LRA formula. We now eliminate
quantifiers right-to-left (this is how we built the CAD as well):

- At the highest level, $R^n$, assign each cell a truth value by evaluating a
  sample point on the quantifier-free part of the formula.
- At level $n-1$, assign each cell a truth value based on the cells in its stack
  above:
  - For an $\exists$, the cell is true if at least 1 cell in the stack above is
    true.
  - For a $\forall$, the cell is true if all cells in the stack above are true.
- Continue until $R^0$, where we will have a singular truth value.

## Basic Example

Let's start with a basic example first. Take this formula:


$x<2 \land y<3 \land x+y>4$

We first construct our CAD. We moved the CAD-creation to some library code,
which we can simply pass the list of constraints, i.e., $-2+x<0$, $-3+y<0$, and
$-4+x+y>0$.

In [9]:
import duckdb as db
import util.sqlcad as sqlcad

DBNAME = "dbs/cad_qe.db"


with db.connect(DBNAME) as con:
    constraints = [
        ["x<2", -2, 1],
        ["y<3", -3, 0, 1],
        ["x+y>4", -4, 1, 1],
    ]

    sqlcad.create_cad(con, constraints)

Now we need to assign each cell of the $n$-dimensional CAD its truth value based
on the quantifier-free part. We have some one-off code to do this for simple
conjuctions; later we will use a more general approach.

In [10]:
with open("queries/qe/results.sql", 'r', encoding='utf-8') as file:
    results_template = file.read()

def write_results(con, dimensions, constraints):
    calc = "a0"
    for i in range(1, dimensions+1):
        calc += f" + a{i}*x{i}"

    xvals = ", ".join(f"l{i}.x{i}" for i in range(1, dimensions+1))

    compiled_query_parts = []
    for i, constraint in enumerate(constraints):
        calc = f"lc{i}.a0"
        for d in range(1, dimensions+1):
            calc += f" + lc{i}.a{d}*x{d}"

        op = '<' if '<' in constraint[0] else '>'
        compiled_query_parts.append(f"{calc} {op} 0")

    compiled_query = " AND ".join(compiled_query_parts)

    constraint_joins = ""
    for i, constraint in enumerate(constraints):
        constraint_joins += f"JOIN LinearConstraint lc{i} ON lc{i}.description = '{constraint[0]}'\n"

    lift_joins = ""
    for i in range(1, dimensions):
        lift_joins += f"JOIN Lift_Dimension{i+1} l{i+1} ON l{i+1}.base_cell = l{i}.id\n"

    query = results_template.format(
        xvals=xvals,
        highest_lift_alias=f"l{dimensions}",
        compiled_query=compiled_query,
        constraint_joins=constraint_joins,
        lift_joins=lift_joins
    )

    con.sql(f"INSERT INTO Result(cell_id, truth_value) {query}")

with db.connect(DBNAME) as con:
    write_results(con, 2, constraints)

We will try some combinations of quantifiers on this. Let's start with $\exists
x \exists y$. We eliminate quantifiers right-to-left, so $\exists y$ is the
first one. This is basically identical to what we've done before: we simply
check if there is any `true` cell.

We use the `Result` table, an extra table that holds the truth-values of each
$n$-dimensional cell for the quantifier-free formula.

In [11]:
query = """
SELECT * FROM Result WHERE truth_value
"""

with db.connect(DBNAME) as con:
    display(con.sql(query))

┌─────────┬─────────────┐
│ cell_id │ truth_value │
│  int32  │   boolean   │
├─────────┼─────────────┤
│      11 │ true        │
└─────────┴─────────────┘

A $\forall y$ could look as follows:

In [12]:
query = """
SELECT NOT EXISTS(SELECT 1 FROM Result WHERE NOT truth_value) AS forall_holds
"""

with db.connect(DBNAME) as con:
    display(con.sql(query))

┌──────────────┐
│ forall_holds │
│   boolean    │
├──────────────┤
│ false        │
└──────────────┘

Of alternatively with a `BOOL_AND`

In [13]:
query = """
SELECT BOOL_AND(truth_value) AS forall_holds
FROM Result
"""

with db.connect(DBNAME) as con:
    display(con.sql(query))

┌──────────────┐
│ forall_holds │
│   boolean    │
├──────────────┤
│ false        │
└──────────────┘

Now, to actually eliminate all quantifiers ($\exists x \exists y$), we check for
each 1-dimensional cell if its stack above has at least one true value. We do
this by doing a `BOOL_OR` on `Lift_Dimension2` and grouping on the base cell ID.
This eliminates $\exists y$.

Then, to eliminate $\exists x$, we do the same again on the results of the
previous expression.

In [14]:
eliminate_y = """
SELECT l2.base_cell, BOOL_OR(r.truth_value) AS truth_value
FROM Result r
JOIN Lift_Dimension2 l2 ON l2.id = r.cell_id
GROUP BY l2.base_cell
"""

eliminate_x = f"""
SELECT BOOL_OR(r.truth_value) AS truth_value
FROM ({eliminate_y}) r
JOIN Lift_Dimension1 l1 ON l1.id = r.base_cell
"""

with db.connect(DBNAME) as con:
    print("All 1D cells after eliminating 'exists y':")
    display(con.sql(eliminate_y))

    print("Single 0D cell after eliminating 'exists x':")
    display(con.sql(eliminate_x))

All 1D cells after eliminating 'exists y':


┌───────────┬─────────────┐
│ base_cell │ truth_value │
│   int32   │   boolean   │
├───────────┼─────────────┤
│         1 │ false       │
│         2 │ false       │
│         3 │ true        │
│         4 │ false       │
│         5 │ false       │
└───────────┴─────────────┘

Single 0D cell after eliminating 'exists x':


┌─────────────┐
│ truth_value │
│   boolean   │
├─────────────┤
│ true        │
└─────────────┘

Combining `BOOL_AND` and `BOOL_OR` until we reach $R^0$ will be the general
approach for any number of quantifiers.

## Quantifier Alternation

Let's take another example, with actual quantifier alternation:

$\forall x \exists y \exists z . (y < x \land x < z)$

First, create the CAD again.

In [15]:
with db.connect(DBNAME) as con:
    constraints = [
        ["y<x", 0, -1, 1],
        ["x<z", 0, 1, 0, -1],
    ]

    sqlcad.create_cad(con, constraints)
    write_results(con, 3, constraints)

Now we eliminate eah quantifier one-by-one.

In [16]:
eliminate_z = """
SELECT l3.base_cell, BOOL_OR(r.truth_value) AS truth_value
FROM Result r
JOIN Lift_Dimension3 l3 ON l3.id = r.cell_id
GROUP BY l3.base_cell
"""

eliminate_y = f"""
SELECT l2.base_cell, BOOL_OR(r.truth_value) AS truth_value
FROM ({eliminate_z}) r
JOIN Lift_Dimension2 l2 ON l2.id = r.base_cell
GROUP BY l2.base_cell
"""

query = f"""
SELECT BOOL_AND(truth_value) AS truth_value
FROM ({eliminate_y} r
JOIN Lift_Dimension1 l1 ON l1.id = r.base_cell
"""

with db.connect(DBNAME) as con:
    print("Original cells:")
    display(con.sql("SELECT * FROM Result"))

    print("All 2D cells after eliminating 'exists z':")
    display(con.sql(eliminate_z))

    print("All 1D cells after eliminating 'exists y':")
    display(con.sql(eliminate_y))

    print("The 0D cell after eliminating 'forall x':")
    display(con.sql(eliminate_x))

Original cells:


┌─────────┬─────────────┐
│ cell_id │ truth_value │
│  int32  │   boolean   │
├─────────┼─────────────┤
│       1 │ false       │
│       2 │ false       │
│       3 │ true        │
│       4 │ false       │
│       5 │ false       │
│       6 │ false       │
│       7 │ false       │
│       8 │ false       │
│       9 │ false       │
└─────────┴─────────────┘

All 2D cells after eliminating 'exists z':


┌───────────┬─────────────┐
│ base_cell │ truth_value │
│   int32   │   boolean   │
├───────────┼─────────────┤
│         1 │ true        │
│         2 │ false       │
│         3 │ false       │
└───────────┴─────────────┘

All 1D cells after eliminating 'exists y':


┌───────────┬─────────────┐
│ base_cell │ truth_value │
│   int32   │   boolean   │
├───────────┼─────────────┤
│         1 │ true        │
└───────────┴─────────────┘

The 0D cell after eliminating 'forall x':


┌─────────────┐
│ truth_value │
│   boolean   │
├─────────────┤
│ true        │
└─────────────┘

These were very basic examples. In the next notebook we walk through our general
approach and work out some more advanced formulas. In the general approach, we
will also return a satisfying model if one exists.